# Semantic segmentation classes
0:Background
1:Neoplastic
2:Inflammatory
3:Connective
4:Dead
5:Epithelial

change in dice metrics

In [1]:
import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import os, gc
import torch
#from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset, DataLoader,TensorDataset
from datasets import load_dataset

In [2]:
import os


import torch.nn.functional as F

import torch.optim as optim

from PIL import Image

In [3]:
import os
import sys

from datasets import load_dataset
sys.path.append('/kaggle/input/datasets/iristhomas2507/utility')
import utils as u

In [4]:
import os

print(os.listdir('/kaggle/input'))

['datasets']


In [5]:

class SegDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.images[idx], dtype=torch.float32),
            torch.tensor(self.masks[idx], dtype=torch.long)
        )

In [6]:


def preprocess_all_folds(dataset, mode_inp):

    folds = []

    for fold_name in ["fold1", "fold2", "fold3"]:
        print(f"\nProcessing {fold_name}")
        print("Samples:", len(dataset[fold_name]))

        imgs, msks = process_fold(
            dataset[fold_name],
            fold_name,
            mode_inp
        )

        folds.append((imgs, msks))

    return folds


def process_fold(hf_dataset, fold_name, mode_inp):

    images, masks = [], []

    for sample in hf_dataset:

        img = preprocess_image(sample["image"])
        msk = preprocess_mask(
            sample["instances"],
            labels=sample["categories"],
            mode=mode_inp
        )

        images.append(img)
        masks.append(msk)

    # Convert to numpy arrays
    images = np.stack(images)
    masks  = np.stack(masks)

    print(f"{fold_name} done:")
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    print("Mask values:", np.unique(masks))

    return images, masks

In [7]:
def dice_loss(pred, target, num_classes, eps=1e-6):

    pred = F.softmax(pred, dim=1)

    target_onehot = F.one_hot(target, num_classes=num_classes)
    target_onehot = target_onehot.permute(0, 3, 1, 2).float()

    intersection = torch.sum(pred * target_onehot, dim=(0,2,3))
    union = torch.sum(pred + target_onehot, dim=(0,2,3))

    dice = (2 * intersection + eps) / (union + eps)

    return 1 - dice[1:].mean()   # ignore background

In [8]:
"""
def dice_score(pred, target, num_classes, eps=1e-6):

    pred = torch.argmax(pred, dim=1)
    dice = 0.0

    for cls in range(num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice += (2 * intersection + eps) / (union + eps)

    return dice / num_classes
"""

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os


# =========================
# Dice utilities (FIXED)
# =========================

def dice_stats(pred, target, num_classes, eps=1e-6):
    pred = torch.argmax(pred, dim=1)

    total_intersection = 0.0
    total_union = 0.0

    for c in range(1, num_classes):  # ignore background
        pred_c = (pred == c)
        target_c = (target == c)

        intersection = (pred_c & target_c).sum().float()
        union = pred_c.sum() + target_c.sum()

        if union == 0:
            continue

        total_intersection += intersection
        total_union += union

    return total_intersection, total_union


def compute_dice_from_stats(total_intersection, total_union, eps=1e-6):
    return (2 * total_intersection + eps) / (total_union + eps)


def per_class_dice(pred, target, num_classes, eps=1e-6):
    pred = torch.argmax(pred, dim=1)
    class_dices = {}

    for c in range(1, num_classes):
        pred_c = (pred == c)
        target_c = (target == c)

        intersection = (pred_c & target_c).sum().float()
        union = pred_c.sum() + target_c.sum()

        if union == 0:
            class_dices[c] = None
            continue

        dice = (2 * intersection + eps) / (union + eps)
        class_dices[c] = dice.item()

    return class_dices


# =========================
# TRAIN FUNCTION (FIXED)
# =========================

def train_model(
    train_dataset,
    val_dataset,
    model,
    fold_idx,
    num_classes=6,
    epochs=10,
    batch_size=8,
    lr=1e-3,
    save_predictions=False,
    save_dir="predictions",
    file_suffix="trial"
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_dice = 0.0

    if save_predictions:
        os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):

        # =========================
        # TRAIN
        # =========================
        model.train()
        total_loss = 0

        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()

            outputs = model(imgs)

            ce = criterion(outputs, masks)
            dl = dice_loss(outputs, masks, num_classes)  # make sure this is SOFT dice

            loss = ce + dl

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # =========================
        # VALIDATION (FIXED)
        # =========================
        model.eval()

        total_intersection = 0.0
        total_union = 0.0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)

                inter, union = dice_stats(outputs, masks, num_classes)

                total_intersection += inter
                total_union += union

        avg_dice = compute_dice_from_stats(
            total_intersection, total_union
        ).item()

        print(f"[Fold {fold_idx}] Epoch {epoch+1}/{epochs}")
        print(f"Loss: {avg_loss:.4f} | Val Dice (no BG): {avg_dice:.4f}")

        # =========================
        # SAVE BEST MODEL (NOW CORRECT)
        # =========================
        if avg_dice > best_dice:
            best_dice = avg_dice
            print("New best model")

            torch.save(
                model.state_dict(),
                f"best_model_fold{fold_idx}.pth"
            )

            # =========================
            # OPTIONAL: Save predictions
            # =========================
            if save_predictions:
                imgs, masks = next(iter(val_loader))
                imgs, masks = imgs.to(device), masks.to(device)

                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)

                for i in range(min(5, preds.shape[0])):
                    save_mask(
                        preds[i],
                        os.path.join(
                            save_dir,
                            f"fold{fold_idx}_pred_{i}_{file_suffix}.png"
                        )
                    )

                    save_mask(
                        masks[i],
                        os.path.join(
                            save_dir,
                            f"fold{fold_idx}_gt_{i}_{file_suffix}.png"
                        )
                    )

        # =========================
        # OPTIONAL DEBUG (HIGHLY USEFUL)
        # =========================
        # Print per-class dice for last batch
        print("Per-class Dice:", per_class_dice(outputs, masks, num_classes))

    return best_dice

In [10]:
import numpy as np

def run_cross_validation(
    processed_folds,
    num_classes=6,
    batch_size=8
):
    """
    Performs K-fold cross-validation (K = number of folds in processed_folds)

    processed_folds: list of tuples -> [(imgs, masks), (imgs, masks), ...]
    Each tuple corresponds to one fold

    Returns:
        fold_scores: list of best Dice scores for each fold
    """

    fold_scores = []

    num_folds = len(processed_folds)

    # Loop over each fold → use it as validation once
    for fold_idx in range(num_folds):

        print(f"\n{'='*60}")
        print(f"Fold {fold_idx} as Validation")
        print(f"{'='*60}")

        # ---------------------------
        # 1. Split data
        # ---------------------------

        # Validation data = current fold
        val_imgs, val_masks = processed_folds[fold_idx]

        # Training data = all other folds
        train_imgs = np.concatenate([
            processed_folds[i][0]
            for i in range(num_folds) if i != fold_idx
        ])

        train_masks = np.concatenate([
            processed_folds[i][1]
            for i in range(num_folds) if i != fold_idx
        ])

        print("Train:", train_imgs.shape, train_masks.shape)
        print("Val  :", val_imgs.shape, val_masks.shape)

        # ---------------------------
        # 2. Create datasets
        # ---------------------------

        train_dataset = SegDataset(train_imgs, train_masks)
        val_dataset   = SegDataset(val_imgs, val_masks)

        # ---------------------------
        # 3. Initialize model
        # ---------------------------
        # IMPORTANT: new model for each fold (no weight sharing)

        model = u.unet(num_classes)

        # ---------------------------
        # 4. Train model
        # ---------------------------

        best_dice = train_model(
            train_dataset,
            val_dataset,
            model,
            fold_idx,
            num_classes=num_classes,
            batch_size=batch_size
        )

        # Store performance
        fold_scores.append(best_dice)

        print(f"Fold {fold_idx} Best Dice: {best_dice:.4f}")

    # ---------------------------
    # 5. Final CV result
    # ---------------------------

    mean_score = np.mean(fold_scores)
    std_score  = np.std(fold_scores)

    print("\n" + "="*60)
    print(f"FINAL CV RESULT: {mean_score:.4f} ± {std_score:.4f}")
    print("="*60)

    return fold_scores

In [11]:
def preprocess_mask(instances, labels=None, mode="binary", num_classes=6):
    instances = np.array(instances)

   
    # CASE 1: EMPTY (no nuclei)
  
    if instances.size == 0 or instances.ndim == 1:
        if mode == "binary":
            mask = np.zeros((256, 256), dtype=np.uint8)
        else:  # semantic
            mask = np.zeros((256, 256), dtype=np.int64)

    else:
        H, W = instances.shape[1], instances.shape[2]


        # BINARY SEGMENTATION 
        
        if mode == "binary":
            mask = np.any(instances > 0, axis=0)
            mask = mask.astype(np.uint8)

       
        # SEMANTIC SEGMENTATION 
        
        elif mode == "semantic":
            if labels is None:
                raise ValueError("labels required for semantic segmentation")

            mask = np.zeros((H, W), dtype=np.int64)

            for i in range(len(instances)):
                instance_mask = instances[i] > 0   # ensure binary
                class_id = int(labels[i]) + 1      # shift for background=0

                # safer overwrite (handles overlap slightly better)
                mask[instance_mask] = np.maximum(
                    mask[instance_mask],
                    class_id
                )

        else:
            raise ValueError("mode must be 'binary' or 'semantic'")

    
    # SAFETY CHECK
    
    if mask.shape != (256, 256):
        print("\nUnexpected mask shape")
        print("Raw shape:", instances.shape)
        print("After merge:", mask.shape)
        print("Unique:", np.unique(mask))

        if mode == "binary":
            mask = np.zeros((256, 256), dtype=np.uint8)
        else:
            mask = np.zeros((256, 256), dtype=np.int64)


    # CHANNEL HANDLING
   
    if mode == "binary":
        # (1, H, W)
        mask = np.expand_dims(mask, axis=0)

    elif mode == "semantic":
        #CrossEntropyLoss expects (H, W)
        pass

    return mask

In [12]:
def preprocess_image(image):
    #print("\n[IMAGE PREPROCESSING]")
    image = np.array(image)
    #print("Original shape:", image.shape)
    #print("Original dtype:", image.dtype)

    # Normalize to [0,1]
    image = image.astype(np.float32) / 255.0

    # Convert HWC → CHW
    image = np.transpose(image, (2, 0, 1))

    #print("Processed shape (CHW):", image.shape)
    #print("Min/Max values:", image.min(), image.max())

    return image

In [13]:
dataset = load_dataset("RationAI/PanNuke")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/fold1-00000-of-00001.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/fold2-00000-of-00001.parquet:   0%|          | 0.00/264M [00:00<?, ?B/s]

data/fold3-00000-of-00001.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

Generating fold1 split:   0%|          | 0/2656 [00:00<?, ? examples/s]

Generating fold2 split:   0%|          | 0/2523 [00:00<?, ? examples/s]

Generating fold3 split:   0%|          | 0/2722 [00:00<?, ? examples/s]

DatasetDict({
    fold1: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2656
    })
    fold2: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2523
    })
    fold3: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2722
    })
})


In [14]:
processed_folds = preprocess_all_folds(dataset, "semantic")




Processing fold1
Samples: 2656
fold1 done:
Images: (2656, 3, 256, 256)
Masks: (2656, 256, 256)
Mask values: [0 1 2 3 4 5]

Processing fold2
Samples: 2523
fold2 done:
Images: (2523, 3, 256, 256)
Masks: (2523, 256, 256)
Mask values: [0 1 2 3 4 5]

Processing fold3
Samples: 2722
fold3 done:
Images: (2722, 3, 256, 256)
Masks: (2722, 256, 256)
Mask values: [0 1 2 3 4 5]


In [15]:
#run_cross_validation(processed_folds)

In [ ]:
scores = run_cross_validation(
    processed_folds,
    num_classes=6,
    batch_size=8
)


Fold 0 as Validation
Train: (5245, 3, 256, 256) (5245, 256, 256)
Val  : (2656, 3, 256, 256) (2656, 256, 256)
[Fold 0] Epoch 1/10
Loss: 1.4669 | Val Dice (no BG): 0.2832
New best model
Per-class Dice: {1: 0.323994517326355, 2: 0.38297224044799805, 3: 0.026996519416570663, 4: 2.610966109983792e-09, 5: None}


In [ ]:
print("\nFold Scores:", scores)
print("Mean Dice:", np.mean(scores))
print("Std Dev :", np.std(scores))